# 23 — Sequence Model (GRU over last-k fights)

**Goal:** A *stretch* notebook. All prior models use aggregated career stats. This one asks: does a **sequence encoder over each fighter's recent fight history** beat the tabular HGB on `Win_A`?

**Design:**
- For each fight, pull the last `K=8` fights for each corner (padded when fewer).
- Per-fight feature vector = per-minute rates + win + method-bucket one-hot + opponent's Elo + days-since.
- Two-corner siamese-style GRU: encode A's sequence, encode B's sequence, concatenate with contextual features (weight-class, physical, ratings), and predict `Win_A`.

**Baselines on identical test events:** HGB (`full` from NB 19), LR (`full`).

**Note:** This notebook is intentionally compact and doesn't attempt to beat the state-of-the-art. It demonstrates a viable sequence model and reports metrics; if log loss / AUC on test is at or slightly below HGB, the thesis gains a "we tried a more expressive model and tabular wins for this data size" paragraph (a common and honest result in sports modeling).

In [14]:
# ========================================================================
# NOTEBOOK_CODE_HEADER v1
# File: 23_sequence_model.ipynb | cell 1
# Section: Build per-fighter fight sequences
# ========================================================================
import os, sys, json, math, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('../scripts'))
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss, accuracy_score
import matchup_utils as mu

DATA = '../data/processed'; RAW = '../data/raw'
m = pd.read_csv(f'{DATA}/ufc_matchup_features.csv', low_memory=False)
with open(f'{DATA}/ufc_feature_groups.json') as f:
    groups = json.load(f)
fights = pd.read_csv(f'{DATA}/ufc_fight_stats_cleaned.csv')
ev = mu.load_event_dates(f'{RAW}/raw_events.csv')
fights = fights.merge(ev.rename(columns={'Event_Id': 'Event_Id_x'}), on='Event_Id_x', how='left')

K = 8  # sequence length per fighter

# Per-fight feature vector for history (14-D)
def row_feats(row):
    s = float(row['total_fight_seconds']) or 1.0
    return np.array([
        (row.get('Sig_Str_Landed') or 0) / (s / 60),
        (row.get('Sig_Str_Att') or 0) / (s / 60),
        (row.get('Takedowns_Landed') or 0) / (s / 60),
        (row.get('Takedowns_Att') or 0) / (s / 60),
        (row.get('Sub_Attempts') or 0) / (s / 60),
        (row.get('Control_Seconds') or 0) / s,
        (row.get('Knockdowns') or 0) / (s / 60),
        (row.get('Distance_Strikes_Landed') or 0) / max(1.0, (row.get('Sig_Str_Landed') or 1)),
        (row.get('Clinch_Strikes_Landed') or 0) / max(1.0, (row.get('Sig_Str_Landed') or 1)),
        (row.get('Ground_Strikes_Landed') or 0) / max(1.0, (row.get('Sig_Str_Landed') or 1)),
        float(row.get('Won') == 1),
        float('KO' in str(row.get('Method', '')).upper() or 'TKO' in str(row.get('Method', '')).upper()),
        float(str(row.get('Method', '')).upper().startswith('SUB')),
        s,
    ], dtype=np.float32)

# Group each fighter's fights by date
fights['_k'] = list(map(row_feats, fights.to_dict('records')))
fights_sorted = fights.sort_values(['Fighter', 'Event_Date']).reset_index(drop=True)
hist = {f: list(g['_k'].values) for f, g in fights_sorted.groupby('Fighter', sort=False)}
print(f'Sequences for {len(hist)} fighters.')

Sequences for 2637 fighters.


In [15]:
# ========================================================================
# NOTEBOOK_CODE_HEADER v1
# File: 23_sequence_model.ipynb | cell 2
# Section: For each fight, grab the K most recent *pre-fight* rows per corner
# ========================================================================
# Pre-compute cumulative fight indices per fighter for quick lookup
fidx = {f: list(g['Fight_Id'].values) for f, g in fights_sorted.groupby('Fighter', sort=False)}
fdate = {f: list(g['Event_Date'].values) for f, g in fights_sorted.groupby('Fighter', sort=False)}

def history_tensor(fighter, this_fight_date, K=8):
    seqs = hist.get(fighter, [])
    dates = fdate.get(fighter, [])
    # keep only fights strictly before this_fight_date
    idx = [i for i, d in enumerate(dates) if pd.notna(d) and pd.notna(this_fight_date) and d < this_fight_date]
    prior = [seqs[i] for i in idx[-K:]]
    if not prior:
        return np.zeros((K, 14), dtype=np.float32), 0
    arr = np.zeros((K, 14), dtype=np.float32)
    arr[-len(prior):] = prior
    return arr, len(prior)

# Build tensors for each fight row in unified features
base = m[['Fight_Id', 'Event_Date', 'Fighter_A', 'Fighter_B', 'Win_A', 'split']].copy()
base['Event_Date'] = pd.to_datetime(base['Event_Date'])
A = np.zeros((len(base), K, 14), dtype=np.float32)
B = np.zeros((len(base), K, 14), dtype=np.float32)
la = np.zeros(len(base), dtype=np.int32)
lb = np.zeros(len(base), dtype=np.int32)
for i, r in enumerate(base.itertuples(index=False)):
    A[i], la[i] = history_tensor(r.Fighter_A, r.Event_Date, K)
    B[i], lb[i] = history_tensor(r.Fighter_B, r.Event_Date, K)
print(f'A tensor {A.shape}, avg history len A={la.mean():.1f}, B={lb.mean():.1f}')

# Sanitize any inf / nan that may have leaked from row_feats division
A = np.nan_to_num(A, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
B = np.nan_to_num(B, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

# Contextual features (tabular side-channel): a compact slice of the unified table
CTX_COLS = [c for c in (
    [f'delta_{c}' for c in ['Sig_Str_PM_Z', 'Takedown_Att_PM_Z', 'Sub_Att_PM_Z', 'Control_Ratio_Z']]
    + ['elo_diff', 'glicko_diff', 'glicko_phi_sum']
    + [c for c in m.columns if c.startswith('wc_')]
    + ['delta_Height_In', 'delta_Reach_In', 'southpaw_vs_orthodox']
) if c in m.columns]
CTX = m[CTX_COLS].fillna(0).values.astype(np.float32)
CTX = np.nan_to_num(CTX, nan=0.0, posinf=0.0, neginf=0.0)
print(f'CTX shape: {CTX.shape}, features: {len(CTX_COLS)}')

A tensor (8482, 8, 14), avg history len A=4.6, B=3.6
CTX shape: (8482, 25), features: 25


In [16]:
# ========================================================================
# NOTEBOOK_CODE_HEADER v1
# File: 23_sequence_model.ipynb | cell 3
# Section: Define and train siamese GRU model
# ========================================================================
try:
    import torch
    import torch.nn as nn
except Exception as e:
    print('PyTorch not available:', e)
    torch = None

if torch is not None:
    device = 'cpu'  # the dataset is small; CPU is fine

    class SiameseGRU(nn.Module):
        def __init__(self, in_dim=14, hid=32, ctx_dim=0, drop=0.2):
            super().__init__()
            self.gru = nn.GRU(in_dim, hid, batch_first=True)
            self.head = nn.Sequential(
                nn.Linear(2 * hid + ctx_dim, 64),
                nn.ReLU(),
                nn.Dropout(drop),
                nn.Linear(64, 1),
            )

        def forward(self, xa, xb, ctx):
            _, ha = self.gru(xa); _, hb = self.gru(xb)
            ha, hb = ha.squeeze(0), hb.squeeze(0)
            return self.head(torch.cat([ha, hb, ctx], dim=1)).squeeze(-1)

    tr = base.split == 'train'
    va = base.split == 'val'
    te = base.split == 'test'
    Xa_tr = torch.tensor(A[tr.values]); Xb_tr = torch.tensor(B[tr.values])
    Xa_va = torch.tensor(A[va.values]); Xb_va = torch.tensor(B[va.values])
    Xa_te = torch.tensor(A[te.values]); Xb_te = torch.tensor(B[te.values])
    Cc_tr = torch.tensor(CTX[tr.values]); Cc_va = torch.tensor(CTX[va.values]); Cc_te = torch.tensor(CTX[te.values])
    y_tr = torch.tensor(base.loc[tr, 'Win_A'].values.astype(np.float32))
    y_va = torch.tensor(base.loc[va, 'Win_A'].values.astype(np.float32))
    y_te = torch.tensor(base.loc[te, 'Win_A'].values.astype(np.float32))

    torch.manual_seed(42)
    model = SiameseGRU(in_dim=14, hid=32, ctx_dim=CTX.shape[1]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    loss_fn = nn.BCEWithLogitsLoss()

    # Symmetrize training by pairing each row with a corner-swap (label flip)
    delta_idx = torch.tensor(
        [i for i, c in enumerate(CTX_COLS) if c.startswith('delta_')], dtype=torch.long)

    def swap_train_batch(xa, xb, ctx, y):
        mask = torch.rand(len(y)) < 0.5
        xa2, xb2, y2, ctx2 = xa.clone(), xb.clone(), y.clone(), ctx.clone()
        xa2[mask], xb2[mask] = xb[mask], xa[mask]
        y2[mask] = 1 - y[mask]
        if len(delta_idx) > 0 and mask.any():
            # Proper advanced-indexing update: build submatrix then write it back.
            sub = ctx2[mask]
            sub[:, delta_idx] = -sub[:, delta_idx]
            ctx2[mask] = sub
        return xa2, xb2, ctx2, y2

    best_va = None
    best_state = None
    BS = 256
    for epoch in range(25):
        model.train()
        perm = torch.randperm(len(y_tr))
        losses = []
        for i in range(0, len(perm), BS):
            idx = perm[i:i+BS]
            xa, xb, cc, yy = Xa_tr[idx], Xb_tr[idx], Cc_tr[idx], y_tr[idx]
            xa, xb, cc, yy = swap_train_batch(xa, xb, cc, yy)
            logits = model(xa, xb, cc)
            loss = loss_fn(logits, yy)
            opt.zero_grad(); loss.backward(); opt.step()
            losses.append(loss.item())
        model.eval()
        with torch.no_grad():
            p_va = torch.sigmoid(model(Xa_va, Xb_va, Cc_va)).cpu().numpy()
        va_ll = log_loss(y_va.cpu().numpy(), np.clip(p_va, 1e-6, 1-1e-6))
        va_auc = roc_auc_score(y_va.cpu().numpy(), p_va)
        if (best_va is None) or (va_ll < best_va):
            best_va = va_ll
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        if epoch % 5 == 0 or epoch == 24:
            print(f'epoch {epoch:02d} | train_loss {np.mean(losses):.4f} | val_ll {va_ll:.4f} | val_auc {va_auc:.4f}')

    model.load_state_dict(best_state)
    with torch.no_grad():
        p_te = torch.sigmoid(model(Xa_te, Xb_te, Cc_te)).cpu().numpy()
    y_te_np = y_te.cpu().numpy()
    print('\nSiamese GRU test:')
    print(f'  AUC  : {roc_auc_score(y_te_np, p_te):.4f}')
    print(f'  Brier: {brier_score_loss(y_te_np, np.clip(p_te, 1e-6, 1-1e-6)):.4f}')
    print(f'  LogLL: {log_loss(y_te_np, np.clip(p_te, 1e-6, 1-1e-6)):.4f}')
    print(f'  Acc  : {accuracy_score(y_te_np, (p_te >= 0.5).astype(int)):.4f}')

epoch 00 | train_loss 3.4526 | val_ll 1.5142 | val_auc 0.4571
epoch 05 | train_loss 1.2912 | val_ll 0.6886 | val_auc 0.5474
epoch 10 | train_loss 0.7712 | val_ll 0.6779 | val_auc 0.5720
epoch 15 | train_loss 0.6948 | val_ll 0.6914 | val_auc 0.5501
epoch 20 | train_loss 0.6831 | val_ll 0.6872 | val_auc 0.5649
epoch 24 | train_loss 0.6755 | val_ll 0.6776 | val_auc 0.5757

Siamese GRU test:
  AUC  : 0.5758
  Brier: 0.2429
  LogLL: 0.6788
  Acc  : 0.5649


In [13]:
# ========================================================================
# NOTEBOOK_CODE_HEADER v1
# File: 23_sequence_model.ipynb | cell 4
# Section: HGB baseline on 'full' feature set (identical test events)
# ========================================================================
FULL = [c for c in m.columns if c.startswith(('delta_pre_', 'mean_pre_', 'A_pk', 'B_pk', 'delta_pk',
                                              'A_ae', 'B_ae', 'delta_ae', 'A_Hybrid', 'B_Hybrid',
                                              'delta_Hybrid', 'A_Height', 'B_Height', 'delta_Height',
                                              'A_Reach', 'B_Reach', 'delta_Reach', 'wc_', 'A_stance_',
                                              'B_stance_'))] + [c for c in (
    ['elo_diff', 'elo_p_A', 'Elo_A_pre', 'Elo_B_pre',
     'glicko_diff', 'glicko_p_A', 'glicko_phi_sum',
     'southpaw_vs_orthodox', 'p_hmap_A',
     'delta_days_since_last', 'mean_days_since_last',
     'days_since_last_A', 'days_since_last_B']
) if c in m.columns]
FULL = [c for c in FULL if c in m.columns]
tr_df = m[m.split == 'train'].reset_index(drop=True)
te_df = m[m.split == 'test'].reset_index(drop=True)
tr_sym = mu.symmetrize_matchup(tr_df, feat_cols=[], label_col='Win_A')
clf = HistGradientBoostingClassifier(max_iter=400, max_depth=8, learning_rate=0.05,
                                     l2_regularization=0.1, random_state=42)
clf.fit(tr_sym[FULL].values, tr_sym['Win_A'].values)
p_hgb = clf.predict_proba(te_df[FULL].values)[:, 1]
y_te_np2 = te_df['Win_A'].values
print('HGB full test:')
print(f'  AUC  : {roc_auc_score(y_te_np2, p_hgb):.4f}')
print(f'  Brier: {brier_score_loss(y_te_np2, np.clip(p_hgb, 1e-6, 1-1e-6)):.4f}')
print(f'  LogLL: {log_loss(y_te_np2, np.clip(p_hgb, 1e-6, 1-1e-6)):.4f}')
print(f'  Acc  : {accuracy_score(y_te_np2, (p_hgb >= 0.5).astype(int)):.4f}')

HGB full test:
  AUC  : 0.6495
  Brier: 0.2423
  LogLL: 0.6901
  Acc  : 0.6189


### Takeaway
- Sequence models typically **don't beat well-featurized tabular boosting** on MMA-scale data (≈ 8,000 fights). A result showing the GRU near HGB (log loss gap < 0.02) is reasonable for the thesis: it validates the tabular pipeline while demonstrating awareness of deeper approaches.
- If the GRU *does* beat HGB by a statistically meaningful margin, the thesis gains a novel modeling contribution. Either way, the comparison is an honest engineering result.
- Future extensions: Transformer w/ learned positional encodings on fight order, include opponent-quality Elo-at-that-time in each history entry, and inject event-level context (title fight / main card).